# 07 — Player Qualities Pre/Post Transfer

Extract player quality scores before and after transfer for internal transfers.

**Filters:**
- 8 leagues (PL, La Liga, Serie A, Ligue 1, Eredivisie, Bundesliga, Primeira Liga, Super Lig)
- Summer windows 2020–2024 (to_season = 2020–2024)
- Internal transfers only (from_competition == to_competition)
- Same position pre/post (from_position == to_position)
- Minimum 600 minutes played both pre and post transfer

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

DATA_ROOT = '/Users/jorgepadilla/Documents/Documents - Jorge\u2019s MacBook Air/thesis_data'
master = pd.read_parquet(f'{DATA_ROOT}/processed_data/master_dataset/transfers_model_2018_2025.parquet')
print(f'Master: {master.shape}')

## 1. Apply Filters

In [ ]:
LEAGUES = {
    364: 'Premier League',
    795: 'La Liga',
    524: 'Serie A',
    412: 'Ligue 1',
    635: 'Eredivisie',
    426: 'Bundesliga',
    707: 'Primeira Liga',
    852: 'Super Lig',
}

SUMMER_WINDOWS = [2020, 2021, 2022, 2023, 2024]  # to_season values
MIN_MINUTES = 600

# The 20 Twelve player qualities
PLAYER_QUALITIES = [
    'Active defence', 'Aerial threat', 'Box threat', 'Chance prevention',
    'Composure', 'Defensive heading', 'Dribbling', 'Effectiveness',
    'Finishing', 'Hold-up play', 'Intelligent defence', 'Involvement',
    'Passing quality', 'Poaching', 'Pressing', 'Progression',
    'Providing teammates', 'Run quality', 'Territorial dominance', 'Winning duels',
]

# Step-by-step filtering
n0 = len(master)

# 1. 8 leagues only
df = master[master['from_competition'].isin(LEAGUES.keys())].copy()
n1 = len(df)

# 2. Internal transfers
df = df[df['from_competition'] == df['to_competition']]
n2 = len(df)

# 3. Summer windows
df = df[df['to_season'].isin(SUMMER_WINDOWS)]
n3 = len(df)

# 4. Same position
df = df[df['from_position'] == df['to_position']]
n4 = len(df)

# 5. Min 600 minutes both sides
df = df[(df['from_Minutes'] >= MIN_MINUTES) & (df['to_Minutes'] >= MIN_MINUTES)]
n5 = len(df)

print('FILTER FUNNEL')
print(f'  Total master:           {n0:>8,}')
print(f'  8 leagues:              {n1:>8,}  (-{n0-n1:,})')
print(f'  Internal only:          {n2:>8,}  (-{n1-n2:,})')
print(f'  Summer 2020-2024:       {n3:>8,}  (-{n2-n3:,})')
print(f'  Same position:          {n4:>8,}  (-{n3-n4:,})')
print(f'  600+ min both sides:    {n5:>8,}  (-{n4-n5:,})')

In [ ]:
# Breakdown by league and window
df['league_name'] = df['from_competition'].map(LEAGUES)

print('Transfers by league:')
print(df['league_name'].value_counts().to_string())

print(f'\nTransfers by window (to_season):')
print(df['to_season'].value_counts().sort_index().to_string())

print(f'\nTransfers by position:')
print(df['from_position'].value_counts().to_string())

print(f'\nCross-table: league x window')
ct = df.groupby(['league_name', 'to_season']).size().unstack(fill_value=0)
ct['TOTAL'] = ct.sum(axis=1)
print(ct.to_string())

## 2. Extract Columns

Keep: IDs, context, and 20 pre-computed player qualities (pre + post).

In [ ]:
# Build column list
id_context_cols = [
    'wy_player_id',
    'from_team_id', 'to_team_id',
    'from_competition', 'league_name',
    'from_season', 'to_season',
    'from_position',  # same as to_position by filter
    'from_Minutes', 'to_Minutes',
    'player_season_age',
]

pre_quality_cols = [f'from_{q}' for q in PLAYER_QUALITIES]
post_quality_cols = [f'to_{q}' for q in PLAYER_QUALITIES]

all_cols = id_context_cols + pre_quality_cols + post_quality_cols

# Verify all columns exist
missing = [c for c in all_cols if c not in df.columns]
if missing:
    print(f'WARNING: Missing columns: {missing}')
else:
    print(f'All {len(all_cols)} columns found')

output = df[all_cols].copy()

# Rename for clarity
rename_map = {}
for q in PLAYER_QUALITIES:
    rename_map[f'from_{q}'] = f'pre_{q}'
    rename_map[f'to_{q}'] = f'post_{q}'
rename_map['from_position'] = 'position'
rename_map['from_Minutes'] = 'pre_minutes'
rename_map['to_Minutes'] = 'post_minutes'
rename_map['from_competition'] = 'competition_id'

output = output.rename(columns=rename_map)

print(f'\nOutput shape: {output.shape}')
print(f'\nColumns:')
for c in output.columns:
    print(f'  {c}')

## 3. Quality Coverage & Null Analysis

In [ ]:
pre_cols = [f'pre_{q}' for q in PLAYER_QUALITIES]
post_cols = [f'post_{q}' for q in PLAYER_QUALITIES]

# Null rates by quality
print('NULL RATES BY QUALITY')
print(f'{"Quality":30s} {"Pre null%":>10s} {"Post null%":>10s}')
print('-' * 52)
for q in PLAYER_QUALITIES:
    pre_null = output[f'pre_{q}'].isna().mean()
    post_null = output[f'post_{q}'].isna().mean()
    flag = ' <<<' if pre_null > 0.05 else ''
    print(f'{q:30s} {pre_null:>9.1%} {post_null:>10.1%}{flag}')

# Null rates by position
print(f'\nNULL RATES BY POSITION (pre-transfer, avg across qualities)')
for pos in output['position'].unique():
    mask = output['position'] == pos
    avg_null = output.loc[mask, pre_cols].isna().mean().mean()
    print(f'  {pos:20s} {avg_null:.1%}  (n={mask.sum()})')

In [ ]:
# Rows with ALL qualities present (both pre and post)
all_present = output[pre_cols + post_cols].notna().all(axis=1).sum()
any_null = output[pre_cols + post_cols].isna().any(axis=1).sum()
print(f'Rows with ALL 40 qualities present: {all_present:,} ({all_present/len(output):.1%})')
print(f'Rows with at least one null:        {any_null:,} ({any_null/len(output):.1%})')

## 4. Quality Distributions & Deltas

In [ ]:
import matplotlib.pyplot as plt

# Compute deltas
for q in PLAYER_QUALITIES:
    output[f'delta_{q}'] = output[f'post_{q}'] - output[f'pre_{q}']

delta_cols = [f'delta_{q}' for q in PLAYER_QUALITIES]

# Summary stats
print('DELTA STATISTICS (post - pre)')
delta_stats = output[delta_cols].describe().T[['mean', 'std', 'min', '25%', '50%', '75%', 'max']]
delta_stats.index = [c.replace('delta_', '') for c in delta_stats.index]
print(delta_stats.round(3).to_string())

In [ ]:
# Histogram of deltas for selected qualities
fig, axes = plt.subplots(4, 5, figsize=(20, 14))
for ax, q in zip(axes.flat, PLAYER_QUALITIES):
    vals = output[f'delta_{q}'].dropna()
    vals = vals[np.isfinite(vals)]
    ax.hist(vals, bins=50, color='steelblue', edgecolor='white', alpha=0.8)
    ax.axvline(0, color='red', ls='--', lw=1)
    ax.axvline(vals.mean(), color='orange', ls='-', lw=1.5)
    ax.set_title(q, fontsize=9)
    ax.tick_params(labelsize=6)

plt.suptitle('Distribution of Quality Deltas (post - pre)\nred=0, orange=mean', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Pre vs Post scatter for a few key qualities
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
sample_q = ['Active defence', 'Finishing', 'Passing quality',
            'Pressing', 'Progression', 'Composure']

for ax, q in zip(axes.flat, sample_q):
    valid = output[[f'pre_{q}', f'post_{q}']].dropna()
    ax.scatter(valid[f'pre_{q}'], valid[f'post_{q}'],
              alpha=0.2, s=8, color='steelblue')
    lims = [valid.min().min() - 0.5, valid.max().max() + 0.5]
    ax.plot(lims, lims, 'r--', lw=1, alpha=0.5)
    ax.set_xlabel('Pre-transfer', fontsize=9)
    ax.set_ylabel('Post-transfer', fontsize=9)
    ax.set_title(q, fontsize=10)
    r = valid.corr().iloc[0, 1]
    ax.text(0.05, 0.95, f'r={r:.3f}', transform=ax.transAxes, fontsize=9, va='top')

plt.suptitle('Pre vs Post Transfer Qualities (same position, same league)', fontsize=12)
plt.tight_layout()
plt.show()

## 5. Save Output

In [ ]:
# Drop delta columns from output (they can be recomputed easily)
save_cols = [c for c in output.columns if not c.startswith('delta_')]
out_df = output[save_cols].reset_index(drop=True)

out_path = f'{DATA_ROOT}/processed_data/model_dataset/v_1/player_qualities_pre_post.parquet'
out_df.to_parquet(out_path, index=False)

print(f'Saved: {out_path}')
print(f'Shape: {out_df.shape}')
print(f'Size: {out_df.memory_usage(deep=True).sum() / 1e6:.1f} MB')
print(f'\n=== SUMMARY ===')
print(f'Transfers: {len(out_df):,}')
print(f'Unique players: {out_df["wy_player_id"].nunique():,}')
print(f'Leagues: {out_df["league_name"].nunique()}')
print(f'Windows: {sorted(out_df["to_season"].unique())}')
print(f'Positions: {sorted(out_df["position"].unique())}')
print(f'Columns: {out_df.shape[1]} (IDs/context: {len(id_context_cols)}, pre qualities: 20, post qualities: 20)')